# MovieNet-318 Scene Segmentation — BaSSL-Inspired Pipeline

Boundary-aware self-supervised learning (BaSSL) pre-training + boundary-head finetuning on frozen CLIP shot features.

See [Mun et al., ACCV 2022](https://arxiv.org/abs/2201.05277). This notebook is a Colab-practical reimplementation for MovieNet-318.

**T4-friendly path:** set `DOWNLOAD_EMBEDDINGS = True` and `DOWNLOAD_KEYFRAMES = False` to use [asmith06/scene-segmentation-embeddings](https://huggingface.co/datasets/asmith06/scene-segmentation-embeddings) (~3 GB) instead of the 50 GB keyframe archive.


## Environment setup

Clones this repository (Colab) or locates it locally, then defines paths under `data/`. Override with environment variables:

- `SCENE_SEG_REPO_URL` — git clone URL (Colab)
- `SCENE_SEG_REPO_DIR` — path to repo root (local Jupyter)

In [ ]:
# --- Environment setup (Colab or local) ---
import os
import subprocess
import sys
from pathlib import Path

# Set your fork URL after publishing, or clone this repo locally.
REPO_URL = os.environ.get(
    "SCENE_SEG_REPO_URL",
    "https://github.com/lwan1/movienet-scene-segmentation.git",
)
REPO_DIR = Path(os.environ.get("SCENE_SEG_REPO_DIR", "/content/movienet-scene-segmentation"))

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
elif not REPO_DIR.exists():
    # Local: assume notebook lives in repo root or notebooks/
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / "data" / "scene318" / "meta" / "split318.json").exists():
            REPO_DIR = candidate
            break
    else:
        raise FileNotFoundError(
            "Could not find repo data/. Clone the repo or set SCENE_SEG_REPO_DIR."
        )

DATA_ROOT = REPO_DIR / "data"
KEYFRAME_ZIP = DATA_ROOT / "keyframes_240p.zip"
KEYFRAME_DIR = DATA_ROOT / "keyframes_240p"

print("REPO_DIR:", REPO_DIR.resolve())
print("DATA_ROOT:", DATA_ROOT.resolve())


## Install dependencies

Installs Python packages from `requirements.txt`, including PyTorch, transformers, and the Hugging Face CLI used for keyframe download.

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)


## Download assets (T4-friendly)

**Recommended on Colab T4:** `DOWNLOAD_EMBEDDINGS = True`, `DOWNLOAD_KEYFRAMES = False`.

Precomputed embeddings (~3 GB) from [asmith06/scene-segmentation-embeddings](https://huggingface.co/datasets/asmith06/scene-segmentation-embeddings) mirror `data/embeddings/` (`visual/`, `subtitle/`, `multimodal/`).

Keyframes (~50 GB) are **optional** — only for building CLIP from scratch or visualization. Uses streaming extraction via `scripts/setup_keyframes.py` (no `extractall`).

In [ ]:
# --- T4-friendly asset toggles ---
DOWNLOAD_EMBEDDINGS = True   # ~3 GB HF download — recommended on Colab T4
DOWNLOAD_KEYFRAMES = False   # ~50 GB — only if encoding CLIP from scratch or visualization
KEYFRAME_SUBSET_ONLY = True  # when DOWNLOAD_KEYFRAMES=True, extract only movies in the current split/subset

import json
import os
import subprocess
import sys

HF_EMBEDDINGS_REPO = "asmith06/scene-segmentation-embeddings"
HF_KEYFRAMES_DATASET = "asmith06/movienet318-keyframes-240p"
HF_KEYFRAMES_FILE = "keyframes_240p.zip"


def keyframes_ready(movie_ids=None) -> bool:
    if not KEYFRAME_DIR.exists():
        return False
    if movie_ids:
        return all((KEYFRAME_DIR / mid).is_dir() for mid in movie_ids)
    return any(KEYFRAME_DIR.glob("tt*"))


def embeddings_ready(movie_ids, modalities=("visual", "subtitle", "multimodal")) -> bool:
    root = DATA_ROOT / "embeddings"
    for modality in modalities:
        mod_dir = root / modality
        if not mod_dir.exists():
            return False
        if not all((mod_dir / f"{mid}.npz").exists() for mid in movie_ids):
            return False
    return True


def download_embeddings_hf(movie_ids=None, modalities=("visual", "subtitle", "multimodal")):
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts" / "download_embeddings_hf.py"),
        "--local-dir",
        str(DATA_ROOT / "embeddings"),
        "--repo",
        HF_EMBEDDINGS_REPO,
        "--modalities",
        *modalities,
    ]
    if movie_ids:
        cmd.extend(["--movie-ids", *movie_ids])
    subprocess.run(cmd, check=True, cwd=REPO_DIR)


def download_and_extract_keyframes(movie_ids=None):
    os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if not KEYFRAME_ZIP.exists():
        print(f"Downloading {HF_KEYFRAMES_FILE} from {HF_KEYFRAMES_DATASET} (~50 GB)...")
        subprocess.run(
            [
                "hf",
                "download",
                HF_KEYFRAMES_DATASET,
                HF_KEYFRAMES_FILE,
                "--repo-type",
                "dataset",
                "--local-dir",
                str(DATA_ROOT),
            ],
            check=True,
        )
    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts" / "setup_keyframes.py"),
        "--zip-path",
        str(KEYFRAME_ZIP),
        "--out-dir",
        str(KEYFRAME_DIR),
    ]
    if movie_ids:
        cmd.extend(["--movie-ids", *movie_ids])
    subprocess.run(cmd, check=True, cwd=REPO_DIR)


def ensure_assets(movie_ids):
    """Download embeddings and/or keyframes for the given movie IDs."""
    if DOWNLOAD_EMBEDDINGS and not embeddings_ready(movie_ids, modalities=("visual",)):
        print(f"Downloading visual embeddings for {len(movie_ids)} movies from HF...")
        subset = movie_ids if len(movie_ids) < 318 else None
        download_embeddings_hf(subset, modalities=("visual",))

    kf_ids = movie_ids if KEYFRAME_SUBSET_ONLY else None
    if DOWNLOAD_KEYFRAMES and not keyframes_ready(kf_ids):
        print("Streaming keyframe extraction (T4-safe, no extractall)...")
        download_and_extract_keyframes(kf_ids)
    elif not DOWNLOAD_KEYFRAMES:
        print("Skipping keyframe download (T4-friendly path).")
    else:
        print("Keyframes already present:", KEYFRAME_DIR)

    if DOWNLOAD_EMBEDDINGS:
        print("Embeddings root:", DATA_ROOT / "embeddings")
    print("KEYFRAME_DIR ready:", keyframes_ready(kf_ids if KEYFRAME_SUBSET_ONLY else None))


## Configuration & hyperparameters

BaSSL-inspired pipeline on **frozen CLIP** features (512-d). Key settings:

| Stage | Parameters |
|-------|------------|
| SSL windows | size 32, stride 16 |
| Transformer CRN | $d{=}256$, 4 heads, 3 layers |
| SSL pretrain | 12 epochs, **train split only** (no scene labels) |
| Finetune | 15 epochs with boundary BCE on train labels |

Set `LIMIT_PER_SPLIT = 3` for a quick end-to-end smoke test.

In [ ]:
import json
import math
import re
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display, Markdown
from PIL import Image
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 5)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

LIMIT_PER_SPLIT: Optional[int] = None  # set to 3 for smoke test
LABEL_POLICY = "lgss"
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
CLIP_DIM = 512

WINDOW_SIZE, WINDOW_STRIDE, MAX_SEQ_LEN = 32, 16, 128
D_MODEL, N_HEAD, N_LAYERS, DROPOUT = 256, 4, 3, 0.1
SSL_EPOCHS, SSL_BATCH_SIZE, SSL_LR = 12, 64, 3e-4
SSL_W_SSM, SSL_W_CGM, SSL_W_PP, SSL_W_MSM = 1.0, 0.5, 1.0, 0.3
MSM_MASK_RATIO = 0.15
FT_EPOCHS, FT_BATCH_SIZE, FT_LR, FT_WEIGHT_DECAY = 15, 4, 1e-4, 1e-4

SOTA_REFERENCE = pd.DataFrame([
    {"method": "Clustering baseline (CLIP)", "f1": 25.0, "ap": 12.0},
    {"method": "BaSSL (published)", "f1": 47.0, "ap": 57.4},
    {"method": "TranS4mer", "f1": 48.4, "ap": 60.8},
    {"method": "MEGA", "f1": 55.3, "ap": 58.6},
    {"method": "Scene-VLM", "f1": 62.1, "ap": 66.8},
])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## Dataset paths

Maps repo directories to MovieNet-318 assets. Writable outputs (embeddings, plots, CSV results) go to `outputs/` so git-tracked `data/` stays read-only.

In [ ]:
# --- Dataset paths (relative to repo) ---
SCENE318_DIR = DATA_ROOT / "scene318"
LABEL_DIR = SCENE318_DIR / "label318"
SHOT_DIR = SCENE318_DIR / "shot_movie318"
SPLIT_PATH = SCENE318_DIR / "meta" / "split318.json"
SUBTITLE_DIR = DATA_ROOT / "subtitle"

# Writable outputs (created outside git-tracked data/)
WORK_ROOT = REPO_DIR / "outputs"
EDA_OUT_DIR = WORK_ROOT / "eda"
EMB_ROOT = DATA_ROOT / "embeddings" if DOWNLOAD_EMBEDDINGS else WORK_ROOT / "embeddings"
RESULTS_DIR = WORK_ROOT / "results"
CKPT_DIR = WORK_ROOT / "checkpoints" / "bassl"

for d in (EDA_OUT_DIR, EMB_ROOT, RESULTS_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

def check_paths():
    required = {"label318": LABEL_DIR, "split318.json": SPLIT_PATH}
    optional = {
        "shot_movie318": SHOT_DIR,
        "subtitles": SUBTITLE_DIR,
        "keyframes": KEYFRAME_DIR,
    }
    print("Required paths:")
    for name, p in required.items():
        print(f"  {'OK' if p.exists() else 'MISSING'}  {name}: {p}")
    print("\nOptional paths:")
    for name, p in optional.items():
        print(f"  {'OK' if p.exists() else 'skip'}  {name}: {p}")

check_paths()


## Output directories

Visual CLIP caches live in `outputs/embeddings/visual/`; model checkpoints in `outputs/checkpoints/bassl/`.

In [ ]:
EMB_ROOT = (DATA_ROOT / "embeddings" / "visual") if DOWNLOAD_EMBEDDINGS else (WORK_ROOT / "embeddings" / "visual")
for d in (CKPT_DIR, EMB_ROOT, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)


## Data loading & label parsing

Loads the official **train / val / test** split (`split318.json`) and per-movie label files.

Each line in `label318/ttXXXX.txt` is `shot_id raw_label`. Under the **LGSS policy**:

$$
y_i = \begin{cases} 1 & \text{if } \text{raw}_i \in \{1, -1\} \\ 0 & \text{otherwise} \end{cases}
$$

Shot timing files align subtitle cues to shots; keyframe paths resolve per-movie image folders.

In [ ]:
def load_split(split_path=SPLIT_PATH):
    with open(split_path, encoding="utf-8") as f:
        split = json.load(f)
    return {k: list(split[k]) for k in ("train", "val", "test")}


def parse_label_file(label_path, policy=LABEL_POLICY):
    rows = []
    with open(label_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = re.split(r"\s+", line)
            if len(parts) < 2:
                continue
            shot_id, raw = int(parts[0]), int(parts[1])
            binary = 1 if raw == -1 else raw if policy == "lgss" else raw
            rows.append({"shot_id": shot_id, "is_boundary": int(binary)})
    return rows


def find_movie_keyframe_dir(movie_id, keyframe_root=KEYFRAME_DIR):
    if keyframe_root is None:
        return None
    keyframe_root = Path(keyframe_root)
    for candidate in (keyframe_root / movie_id, keyframe_root / "240P_frames" / movie_id):
        if candidate.exists():
            return candidate
    matches = [p for p in keyframe_root.rglob(movie_id) if p.is_dir()]
    return matches[0] if matches else None


split = load_split()
if LIMIT_PER_SPLIT is not None:
    split = {k: v[:LIMIT_PER_SPLIT] for k, v in split.items()}

ALL_IDS = split["train"] + split["val"] + split["test"]
MOVIE_TO_SPLIT = {mid: name for name, ids in split.items() for mid in ids}
print(f"Movies: {len(ALL_IDS)} | train/val/test = {len(split['train'])}/{len(split['val'])}/{len(split['test'])}")

ensure_assets(ALL_IDS)


## Frozen CLIP feature extraction

Encodes the middle keyframe of each shot with **CLIP ViT-B/32** and L2-normalizes:

$$
\hat{x}_i = \frac{f_\theta(\text{img}_i)}{\|f_\theta(\text{img}_i)\|}
$$

Features are cached per movie (`.npz`) so SSL and finetune can reload without re-encoding.

In [ ]:
# --- Frozen CLIP feature extraction (cached) ---
from transformers import CLIPModel, CLIPProcessor

clip_model = clip_processor = None
if DOWNLOAD_EMBEDDINGS and all((EMB_ROOT / f"{mid}.npz").exists() for mid in ALL_IDS):
    print("Using precomputed CLIP embeddings from", EMB_ROOT)
elif KEYFRAME_DIR is not None:
    clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(DEVICE)
    clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
    clip_model.eval()
    print("CLIP loaded:", CLIP_MODEL_ID)


def pool_clip_features(features):
    """Handle tensor or ModelOutput return types across transformers versions."""
    if torch.is_tensor(features):
        return features
    if hasattr(features, "image_embeds") and features.image_embeds is not None:
        return features.image_embeds
    if hasattr(features, "pooler_output") and features.pooler_output is not None:
        return features.pooler_output
    if hasattr(features, "last_hidden_state"):
        return features.last_hidden_state[:, 0]
    raise TypeError(f"Unexpected CLIP feature type: {type(features)}")


def encode_movie_clip(movie_id: str) -> np.ndarray:
    """Return L2-normalized CLIP features (n_shots, 512). Uses middle keyframe img_1."""
    cache_path = EMB_ROOT / f"{movie_id}.npz"
    if cache_path.exists():
        return np.load(cache_path)["X"].astype(np.float32)

    label_rows = parse_label_file(LABEL_DIR / f"{movie_id}.txt")
    n = len(label_rows)
    kf_dir = find_movie_keyframe_dir(movie_id)
    if kf_dir is None or clip_model is None:
        raise FileNotFoundError(f"No keyframes for {movie_id}")

    imgs = []
    for i in range(n):
        p = kf_dir / f"shot_{i:04d}_img_1.jpg"
        if not p.exists():
            p = kf_dir / f"shot_{i:04d}_img_0.jpg"
        if p.exists():
            imgs.append(Image.open(p).convert("RGB"))
        else:
            imgs.append(Image.new("RGB", (224, 224), color=(0, 0, 0)))

    feats = []
    batch_size = 64
    with torch.no_grad():
        for start in range(0, n, batch_size):
            batch = imgs[start : start + batch_size]
            inputs = clip_processor(images=batch, return_tensors="pt").to(DEVICE)
            f = pool_clip_features(clip_model.get_image_features(**inputs))
            f = f / f.norm(dim=-1, keepdim=True)
            feats.append(f.cpu().numpy())
    X = np.vstack(feats).astype(np.float32)
    np.savez(cache_path, X=X)
    return X


# Cache all movies (skip existing)
missing_kf = []
for mid in tqdm(ALL_IDS, desc="CLIP cache"):
    if (EMB_ROOT / f"{mid}.npz").exists():
        continue
    try:
        encode_movie_clip(mid)
    except FileNotFoundError:
        missing_kf.append(mid)

if missing_kf:
    print(f"Warning: {len(missing_kf)} movies missing keyframes: {missing_kf[:5]}...")
else:
    print("All CLIP caches ready under", EMB_ROOT)


## Pseudo-boundary discovery

Within each sliding window, finds split index $b$ maximizing pseudo-scene separation on CLIP features (DTW-style heuristic from BaSSL):

$$
\text{score}(b) = d_{\text{inter}}(\mu_A, \mu_B) - \tfrac{1}{4}\big(\text{intra}_A + \text{intra}_B\big)
$$

where $\mu_A, \mu_B$ are mean embeddings of shots $[0,b)$ and $[b,T)$. Shot $b{-}1$ is labeled as a pseudo-boundary. **No ground-truth scene labels are used.**

In [ ]:
# --- Pseudo-boundary discovery (DTW-style split on CLIP embeddings) ---

def discover_pseudo_boundary(feats: np.ndarray, min_group: int = 4) -> int:
    """Split window into two groups with maximum mean-disparity at boundary.

    Returns boundary index b in [min_group, T-min_group): shots [0,b) vs [b,T).
    Shot b-1 is labeled pseudo-boundary (ends pseudo-scene A).
    """
    T = len(feats)
    if T < 2 * min_group:
        return max(1, T // 2)

    X = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-12)
    best_b, best_score = T // 2, -1e9
    for b in range(min_group, T - min_group + 1):
        mean_a = X[:b].mean(axis=0)
        mean_b = X[b:].mean(axis=0)
        mean_a /= np.linalg.norm(mean_a) + 1e-12
        mean_b /= np.linalg.norm(mean_b) + 1e-12
        # transition = inter-scene distance - average intra cohesion penalty
        inter = 1.0 - float(np.dot(mean_a, mean_b))
        intra_a = 1.0 - float(np.mean(X[:b] @ mean_a))
        intra_b = 1.0 - float(np.mean(X[b:] @ mean_b))
        score = inter - 0.25 * (intra_a + intra_b)
        if score > best_score:
            best_score, best_b = score, b
    return best_b


def iter_windows(feats: np.ndarray, window_size=WINDOW_SIZE, stride=WINDOW_STRIDE):
    T = len(feats)
    if T <= window_size:
        yield 0, feats
        return
    for start in range(0, T - window_size + 1, stride):
        yield start, feats[start : start + window_size]
    if (T - window_size) % stride != 0:
        yield T - window_size, feats[-window_size:]


# Quick sanity check on one train movie
_demo = split["train"][0]
_demo_X = encode_movie_clip(_demo)
_s, _w = next(iter_windows(_demo_X))
_pb = discover_pseudo_boundary(_w)
print(f"Demo {_demo}: window len={len(_w)}, pseudo-boundary at local index {_pb} (shot ends scene A)")


## BaSSL-inspired model

A **Transformer encoder** (contextual relation network) maps shot features to contextual representations $h_i$. Heads support:

- **SSM** — shot-to-scene classification (2-way, from pseudo labels)
- **CGM** — contextual group matching (contrastive; same pseudo-scene shots should cluster in CRN space)
- **PP** — pseudo-boundary prediction (BCE)
- **MSM** — masked shot modeling (MSE reconstruction of masked CLIP features)
- **Boundary head** — scalar logit per shot (used after finetune)

In [ ]:
# --- BaSSL-inspired model: contextual Transformer + pretext / boundary heads ---

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class BaSSLModel(nn.Module):
    def __init__(self, input_dim=CLIP_DIM, d_model=D_MODEL, nhead=N_HEAD, nlayers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
        )
        self.pos = PositionalEncoding(d_model, max_len=512)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=nlayers)
        self.ssm_head = nn.Linear(d_model, 2)
        self.pp_head = nn.Linear(d_model, 1)
        self.msm_head = nn.Linear(d_model, input_dim)
        self.boundary_head = nn.Linear(d_model, 1)

    def encode(self, x: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        h = self.input_proj(x)
        h = self.pos(h)
        return self.encoder(h, src_key_padding_mask=key_padding_mask)

    def forward_ssl(self, x: torch.Tensor):
        h = self.encode(x)
        return {
            "context": h,
            "ssm_logits": self.ssm_head(h),
            "pp_logits": self.pp_head(h).squeeze(-1),
            "msm_pred": self.msm_head(h),
        }

    def forward_boundary(self, x: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None):
        h = self.encode(x, key_padding_mask=key_padding_mask)
        return self.boundary_head(h).squeeze(-1)


model = BaSSLModel().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"BaSSLModel parameters: {n_params:,}")


## SSL dataset & losses

Samples sliding windows from **train movies only**. Combined SSL objective:

$$
\mathcal{L} = w_1 \mathcal{L}_{\text{SSM}} + w_2 \mathcal{L}_{\text{CGM}} + w_3 \mathcal{L}_{\text{PP}} + w_4 \mathcal{L}_{\text{MSM}}
$$

Default weights: $(1.0,\; 0.5,\; 1.0,\; 0.3)$. Scene labels from `label318/` are **not** used in this stage.

In [ ]:
# --- SSL dataset: sliding windows from TRAIN movies only (no scene labels) ---

class SSLWindowDataset(Dataset):
    def __init__(self, movie_ids: List[str], samples_per_movie: int = 40):
        self.samples: List[Tuple[str, int]] = []
        self.movie_ids = movie_ids
        for mid in movie_ids:
            try:
                X = encode_movie_clip(mid)
            except FileNotFoundError:
                continue
            windows = list(iter_windows(X))
            if not windows:
                continue
            rng = np.random.default_rng(RANDOM_SEED + hash(mid) % 10000)
            picks = rng.choice(len(windows), size=min(samples_per_movie, len(windows)), replace=False)
            for idx in picks:
                self.samples.append((mid, int(idx)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        mid, widx = self.samples[idx]
        X = encode_movie_clip(mid)
        windows = list(iter_windows(X))
        start, window = windows[widx]
        pb = discover_pseudo_boundary(window)
        T = len(window)
        scene_ids = np.zeros(T, dtype=np.int64)
        scene_ids[pb:] = 1
        pp_labels = np.zeros(T, dtype=np.float32)
        if pb > 0:
            pp_labels[pb - 1] = 1.0
        return {
            "feats": torch.from_numpy(window.copy()),
            "scene_ids": torch.from_numpy(scene_ids),
            "pp_labels": torch.from_numpy(pp_labels),
        }


def collate_ssl(batch):
    feats = torch.stack([b["feats"] for b in batch])
    scene_ids = torch.stack([b["scene_ids"] for b in batch])
    pp_labels = torch.stack([b["pp_labels"] for b in batch])
    return feats, scene_ids, pp_labels


def cgm_loss(context: torch.Tensor, scene_ids: torch.Tensor, temperature: float = 0.1) -> torch.Tensor:
    """Contextual group matching: same pseudo-scene shots should be closer in CRN space."""
    B, T, D = context.shape
    ctx = F.normalize(context, dim=-1)
    losses = []
    for b in range(B):
        labels = scene_ids[b]
        for sid in (0, 1):
            mask = labels == sid
            if mask.sum() < 2:
                continue
            group = ctx[b, mask]
            anchor = group.mean(dim=0, keepdim=True)
            anchor = F.normalize(anchor, dim=-1)
            sim_pos = (group * anchor).sum(dim=-1) / temperature
            other = ctx[b, labels != sid]
            if len(other) == 0:
                continue
            sim_neg = (group @ other.T) / temperature
            for i in range(len(group)):
                logits = torch.cat([sim_pos[i : i + 1], sim_neg[i]])
                target = torch.zeros(1, dtype=torch.long, device=logits.device)
                losses.append(F.cross_entropy(logits.unsqueeze(0), target))
    if not losses:
        return torch.tensor(0.0, device=context.device)
    return torch.stack(losses).mean()


def msm_loss(context: torch.Tensor, target_feats: torch.Tensor, mask_ratio=MSM_MASK_RATIO) -> torch.Tensor:
    B, T, _ = context.shape
    loss = torch.tensor(0.0, device=context.device)
    count = 0
    for b in range(B):
        n_mask = max(1, int(T * mask_ratio))
        idx = torch.randperm(T, device=context.device)[:n_mask]
        pred = model.msm_head(context[b, idx])
        tgt = target_feats[b, idx]
        loss = loss + F.mse_loss(pred, tgt)
        count += 1
    return loss / max(count, 1)


def train_ssl_epoch(model, loader, optimizer) -> Dict[str, float]:
    model.train()
    totals = {"loss": 0.0, "ssm": 0.0, "cgm": 0.0, "pp": 0.0, "msm": 0.0}
    n = 0
    for feats, scene_ids, pp_labels in loader:
        feats = feats.to(DEVICE)
        scene_ids = scene_ids.to(DEVICE)
        pp_labels = pp_labels.to(DEVICE)
        out = model.forward_ssl(feats)

        loss_ssm = F.cross_entropy(out["ssm_logits"].reshape(-1, 2), scene_ids.reshape(-1))
        loss_cgm = cgm_loss(out["context"], scene_ids)
        loss_pp = F.binary_cross_entropy_with_logits(out["pp_logits"], pp_labels)
        loss_msm = msm_loss(out["context"], feats)

        loss = SSL_W_SSM * loss_ssm + SSL_W_CGM * loss_cgm + SSL_W_PP * loss_pp + SSL_W_MSM * loss_msm
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        totals["loss"] += float(loss.item())
        totals["ssm"] += float(loss_ssm.item())
        totals["cgm"] += float(loss_cgm.item())
        totals["pp"] += float(loss_pp.item())
        totals["msm"] += float(loss_msm.item())
        n += 1
    return {k: v / max(n, 1) for k, v in totals.items()}


## SSL pretrain loop

Trains the full model (CRN + pretext heads) for `SSL_EPOCHS` on pseudo-boundary windows. Saves `bassl_ssl_pretrained.pt`. This learns temporal context before any supervised boundary training.

In [ ]:
# --- SSL pretrain loop (train split only, no scene labels) ---

ssl_dataset = SSLWindowDataset(split["train"], samples_per_movie=40)
ssl_loader = DataLoader(
    ssl_dataset,
    batch_size=SSL_BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    collate_fn=collate_ssl,
    num_workers=0,
)
print(f"SSL windows: {len(ssl_dataset)} | batches/epoch: {len(ssl_loader)}")

optimizer_ssl = torch.optim.AdamW(model.parameters(), lr=SSL_LR, weight_decay=1e-4)
scheduler_ssl = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ssl, T_max=SSL_EPOCHS)

ssl_history = []
for epoch in range(1, SSL_EPOCHS + 1):
    stats = train_ssl_epoch(model, ssl_loader, optimizer_ssl)
    scheduler_ssl.step()
    ssl_history.append({"epoch": epoch, **stats})
    print(
        f"SSL epoch {epoch:02d}/{SSL_EPOCHS} | loss={stats['loss']:.4f} "
        f"ssm={stats['ssm']:.3f} cgm={stats['cgm']:.3f} pp={stats['pp']:.3f} msm={stats['msm']:.4f}"
    )

ssl_ckpt = CKPT_DIR / "bassl_ssl_pretrained.pt"
torch.save({"model": model.state_dict(), "config": {"CLIP_DIM": CLIP_DIM, "D_MODEL": D_MODEL}}, ssl_ckpt)
pd.DataFrame(ssl_history).to_csv(RESULTS_DIR / "bassl_ssl_history.csv", index=False)
print("Saved SSL checkpoint:", ssl_ckpt)


Saves the SSL-pretrained weights and training history to `outputs/checkpoints/bassl/` and `outputs/results/`.

In [ ]:
ssl_ckpt = CKPT_DIR / "bassl_ssl_pretrained.pt"
torch.save({"model": model.state_dict(), "config": {"CLIP_DIM": CLIP_DIM, "D_MODEL": D_MODEL}}, ssl_ckpt)
pd.DataFrame(ssl_history).to_csv(RESULTS_DIR / "bassl_ssl_history.csv", index=False)
print("Saved SSL checkpoint:", ssl_ckpt)

In [ ]:
print("model in memory:", "model" in globals())
print("ssl_history in memory:", "ssl_history" in globals())
if "ssl_history" in globals():
    print("ssl epochs completed:", len(ssl_history), "/", SSL_EPOCHS)

## Supervised finetune

Fine-tunes the **boundary head** (full model weights) on **train split** scene labels with weighted BCE:

$$
\mathcal{L}_{\text{BCE}} = - \big[ w_+ y \log \sigma(z) + (1-y) \log(1-\sigma(z)) \big]
$$

with $w_+ \approx 11$ to counter ~8% positive boundary rate. Long movies are processed in overlapping chunks (`MAX_SEQ_LEN=128`).

In [ ]:
# --- Finetune: boundary head on TRAIN labels (chunked full movies) ---

class MovieChunkDataset(Dataset):
    def __init__(self, movie_ids: List[str], use_labels: bool = True, max_len=MAX_SEQ_LEN, stride=None):
        self.chunks = []
        stride = stride or max_len // 2
        for mid in movie_ids:
            try:
                X = encode_movie_clip(mid)
            except FileNotFoundError:
                continue
            labels = None
            if use_labels:
                labels = np.array([r["is_boundary"] for r in parse_label_file(LABEL_DIR / f"{mid}.txt")], dtype=np.float32)
                if len(labels) != len(X):
                    n = min(len(labels), len(X))
                    X, labels = X[:n], labels[:n]
            T = len(X)
            if T <= max_len:
                self.chunks.append({"movie_id": mid, "feats": X, "labels": labels, "offset": 0})
            else:
                for start in range(0, T - max_len + 1, stride):
                    sl = slice(start, start + max_len)
                    self.chunks.append({"movie_id": mid, "feats": X[sl], "labels": labels[sl] if labels is not None else None, "offset": start})
                if (T - max_len) % stride != 0:
                    self.chunks.append({"movie_id": mid, "feats": X[-max_len:], "labels": labels[-max_len:] if labels is not None else None, "offset": T - max_len})

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        c = self.chunks[idx]
        item = {"feats": torch.from_numpy(c["feats"].copy()), "movie_id": c["movie_id"], "offset": c["offset"]}
        if c["labels"] is not None:
            item["labels"] = torch.from_numpy(c["labels"].copy())
        return item


def collate_finetune(batch):
    max_t = max(b["feats"].shape[0] for b in batch)
    B = len(batch)
    d = batch[0]["feats"].shape[1]
    feats = torch.zeros(B, max_t, d)
    labels = torch.zeros(B, max_t)
    mask = torch.ones(B, max_t, dtype=torch.bool)
    for i, b in enumerate(batch):
        t = b["feats"].shape[0]
        feats[i, :t] = b["feats"]
        if "labels" in b:
            labels[i, :t] = b["labels"]
        mask[i, :t] = False
    return feats, labels, mask


# Pos weight for ~8% positive rate
pos_weight = torch.tensor([11.0], device=DEVICE)
criterion_bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")


def train_finetune_epoch(model, loader, optimizer) -> float:
    model.train()
    total, count = 0.0, 0
    for feats, labels, pad_mask in loader:
        feats, labels, pad_mask = feats.to(DEVICE), labels.to(DEVICE), pad_mask.to(DEVICE)
        logits = model.forward_boundary(feats, key_padding_mask=pad_mask)
        loss_mat = criterion_bce(logits, labels)
        valid = ~pad_mask
        loss = loss_mat[valid].mean()
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += float(loss.item())
        count += 1
    return total / max(count, 1)


ft_train_ds = MovieChunkDataset(split["train"], use_labels=True)
ft_train_loader = DataLoader(ft_train_ds, batch_size=FT_BATCH_SIZE, shuffle=True, collate_fn=collate_finetune, num_workers=0)
print(f"Finetune chunks (train): {len(ft_train_ds)}")

optimizer_ft = torch.optim.AdamW(model.parameters(), lr=FT_LR, weight_decay=FT_WEIGHT_DECAY)
scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=FT_EPOCHS)

ft_history = []
for epoch in range(1, FT_EPOCHS + 1):
    loss = train_finetune_epoch(model, ft_train_loader, optimizer_ft)
    scheduler_ft.step()
    ft_history.append({"epoch": epoch, "loss": loss})
    print(f"Finetune epoch {epoch:02d}/{FT_EPOCHS} | bce={loss:.4f}")

ft_ckpt = CKPT_DIR / "bassl_finetuned.pt"
torch.save({"model": model.state_dict()}, ft_ckpt)
pd.DataFrame(ft_history).to_csv(RESULTS_DIR / "bassl_finetune_history.csv", index=False)
print("Saved finetune checkpoint:", ft_ckpt)


## Inference & evaluation

Per-shot boundary score: $\sigma(z_i)$ from the finetuned head. Long movies use overlapping windows with score averaging.

Threshold $\tau$ is tuned on **val** to maximize macro F1, then applied to **test**. A **CLIP adjacent-distance clustering baseline** is included for comparison.

In [ ]:
# --- Inference: per-shot boundary scores for a full movie ---

@torch.no_grad()
def predict_movie_scores(model, movie_id: str) -> np.ndarray:
    X = encode_movie_clip(movie_id)
    T = len(X)
    model.eval()
    if T <= MAX_SEQ_LEN:
        feats = torch.from_numpy(X).unsqueeze(0).to(DEVICE)
        logits = model.forward_boundary(feats)
        scores = torch.sigmoid(logits[0]).cpu().numpy()
        return scores

    scores = np.zeros(T, dtype=np.float32)
    counts = np.zeros(T, dtype=np.float32)
    stride = MAX_SEQ_LEN // 2
    for start in range(0, T - MAX_SEQ_LEN + 1, stride):
        sl = slice(start, start + MAX_SEQ_LEN)
        feats = torch.from_numpy(X[sl]).unsqueeze(0).to(DEVICE)
        logits = model.forward_boundary(feats)
        s = torch.sigmoid(logits[0]).cpu().numpy()
        scores[sl] += s
        counts[sl] += 1
    if (T - MAX_SEQ_LEN) % stride != 0:
        sl = slice(T - MAX_SEQ_LEN, T)
        feats = torch.from_numpy(X[sl]).unsqueeze(0).to(DEVICE)
        logits = model.forward_boundary(feats)
        s = torch.sigmoid(logits[0]).cpu().numpy()
        scores[sl] += s
        counts[sl] += 1
    counts = np.maximum(counts, 1.0)
    return scores / counts


def clustering_baseline_scores(movie_id: str) -> np.ndarray:
    X = encode_movie_clip(movie_id)
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    scores = np.zeros(len(Xn), dtype=np.float32)
    for i in range(1, len(Xn)):
        scores[i] = 1.0 - float(np.dot(Xn[i - 1], Xn[i]))
    return scores


def eval_split(movie_ids, score_fn, threshold: Optional[float] = None):
    f1s, aps, all_y, all_s = [], [], [], []
    for mid in movie_ids:
        y = np.array([r["is_boundary"] for r in parse_label_file(LABEL_DIR / f"{mid}.txt")], dtype=int)
        scores = score_fn(mid)
        n = min(len(y), len(scores))
        y, scores = y[:n], scores[:n]
        if y.sum() == 0:
            continue
        aps.append(average_precision_score(y, scores))
        all_y.append(y)
        all_s.append(scores)
        if threshold is not None:
            f1s.append(f1_score(y, (scores >= threshold).astype(int), zero_division=0))
    if threshold is None and all_y:
        y_cat = np.concatenate(all_y)
        s_cat = np.concatenate(all_s)
        prec, rec, thr = precision_recall_curve(y_cat, s_cat)
        f1_arr = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
        threshold = float(thr[int(np.nanargmax(f1_arr))]) if len(f1_arr) else 0.5
        for y, scores in zip(all_y, all_s):
            f1s.append(f1_score(y, (scores >= threshold).astype(int), zero_division=0))
    return {
        "f1": float(np.mean(f1s)) if f1s else float("nan"),
        "ap": float(np.mean(aps)) if aps else float("nan"),
        "threshold": threshold,
        "n_movies": len(f1s),
    }


# Tune threshold on VAL
val_metrics = eval_split(split["val"], lambda mid: predict_movie_scores(model, mid), threshold=None)
print(f"VAL  | BaSSL finetuned | F1={val_metrics['f1']:.4f} AP={val_metrics['ap']:.4f} thr={val_metrics['threshold']:.4f}")

# Test with val threshold
test_metrics = eval_split(
    split["test"],
    lambda mid: predict_movie_scores(model, mid),
    threshold=val_metrics["threshold"],
)
print(f"TEST | BaSSL finetuned | F1={test_metrics['f1']:.4f} AP={test_metrics['ap']:.4f}")

# Clustering baseline on test for comparison
clust_val = eval_split(split["val"], clustering_baseline_scores, threshold=None)
clust_test = eval_split(split["test"], clustering_baseline_scores, threshold=clust_val["threshold"])
print(f"TEST | CLIP clustering baseline | F1={clust_test['f1']:.4f} AP={clust_test['ap']:.4f}")


## Results summary

Reports F1 and AP (%) on val/test, saves `bassl_test_results.csv` and `bassl_project_summary.json`, and displays published MovieNet-318 reference scores.

In [ ]:
# --- Results table & summary (percent scale, matches published papers) ---

def to_pct(x):
    return round(float(x) * 100, 2)

results = pd.DataFrame([
    {
        "method": "CLIP adjacent-distance (clustering)",
        "split": "test",
        "f1": to_pct(clust_test["f1"]),
        "ap": to_pct(clust_test["ap"]),
        "threshold": clust_val["threshold"],
    },
    {
        "method": "BaSSL-inspired (SSL + finetune)",
        "split": "val",
        "f1": to_pct(val_metrics["f1"]),
        "ap": to_pct(val_metrics["ap"]),
        "threshold": val_metrics["threshold"],
    },
    {
        "method": "BaSSL-inspired (SSL + finetune)",
        "split": "test",
        "f1": to_pct(test_metrics["f1"]),
        "ap": to_pct(test_metrics["ap"]),
        "threshold": val_metrics["threshold"],
    },
])
results.to_csv(RESULTS_DIR / "bassl_test_results.csv", index=False)

print("\n=== Our results (F1 / AP in %) ===")
display(results)

print("\n=== Published MovieNet-318 reference (F1 / AP in %) ===")
display(SOTA_REFERENCE)

summary = {
    "test_f1_pct": to_pct(test_metrics["f1"]),
    "test_ap_pct": to_pct(test_metrics["ap"]),
    "val_f1_pct": to_pct(val_metrics["f1"]),
    "val_ap_pct": to_pct(val_metrics["ap"]),
    "clustering_test_f1_pct": to_pct(clust_test["f1"]),
    "ssl_epochs": SSL_EPOCHS,
    "ft_epochs": FT_EPOCHS,
    "limit_per_split": LIMIT_PER_SPLIT,
    "checkpoints": {"ssl": str(ssl_ckpt), "finetune": str(ft_ckpt)},
}
with open(RESULTS_DIR / "bassl_project_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\nBest test: F1={summary['test_f1_pct']:.2f}%  AP={summary['test_ap_pct']:.2f}%")
print("Saved:", RESULTS_DIR / "bassl_test_results.csv")
print("Saved:", RESULTS_DIR / "bassl_project_summary.json")

## Visualization helpers

Utilities to map boundary flags to scene IDs, load representative keyframes, and render predicted vs. ground-truth scene groupings.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import numpy as np

def labels_to_scene_ids(boundary_flags: np.ndarray) -> np.ndarray:
    """boundary_flags[i]=1 means shot i ends a scene."""
    scene_ids = np.zeros(len(boundary_flags), dtype=int)
    sid = 0
    for i, b in enumerate(boundary_flags):
        scene_ids[i] = sid
        if b == 1:
            sid += 1
    return scene_ids

def load_shot_keyframe(movie_id: str, shot_idx: int) -> Image.Image:
    kf_dir = find_movie_keyframe_dir(movie_id)
    p = kf_dir / f"shot_{shot_idx:04d}_img_1.jpg"
    if not p.exists():
        p = kf_dir / f"shot_{shot_idx:04d}_img_0.jpg"
    if p.exists():
        return Image.open(p).convert("RGB")
    return Image.new("RGB", (240, 135), color=(80, 80, 80))

def predict_boundaries(model, movie_id: str, threshold: float, min_len: int = 4) -> np.ndarray:
    scores = predict_movie_scores_boost(model, movie_id) if "predict_movie_scores_boost" in globals() else predict_movie_scores(model, movie_id)
    if "scores_to_pred" in globals():
        return scores_to_pred(scores, threshold, min_len=min_len)
    return (scores >= threshold).astype(int)

def scene_representative_shots(scene_ids: np.ndarray, max_per_scene: int = 3):
    """Pick up to max_per_scene evenly spaced shots per scene."""
    picks = []
    for sid in np.unique(scene_ids):
        idxs = np.where(scene_ids == sid)[0]
        if len(idxs) <= max_per_scene:
            picks.extend(idxs.tolist())
        else:
            step = max(1, len(idxs) // max_per_scene)
            picks.extend(idxs[::step][:max_per_scene].tolist())
    return picks

## Scene comparison plots

Visualizes predicted vs. ground-truth scene assignments as color-coded keyframe strips for qualitative inspection.

In [ ]:
def plot_scene_comparison(movie_id: str, threshold: float = BEST_THRESHOLD if "BEST_THRESHOLD" in globals() else val_metrics["threshold"], max_scenes: int = 12, shots_per_scene: int = 2):
    label_rows = parse_label_file(LABEL_DIR / f"{movie_id}.txt")
    y_true = np.array([r["is_boundary"] for r in label_rows], dtype=int)

    pred = predict_boundaries(model, movie_id, threshold)
    n = min(len(y_true), len(pred))
    y_true, pred = y_true[:n], pred[:n]

    gt_scenes = labels_to_scene_ids(y_true)
    pred_scenes = labels_to_scene_ids(pred)

    fig, axes = plt.subplots(2, 1, figsize=(14, 5))
    titles = ["Ground truth scenes", "Predicted scenes"]

    for ax, scene_ids, title in zip(axes, [gt_scenes, pred_scenes], titles):
        shown_scenes = 0
        x_cursor = 0
        cmap = plt.cm.tab20
        for sid in range(scene_ids.max() + 1):
            if shown_scenes >= max_scenes:
                break
            idxs = np.where(scene_ids == sid)[0]
            if len(idxs) == 0:
                continue
            pick = idxs[::max(1, len(idxs) // shots_per_scene)][:shots_per_scene]
            for j, shot_idx in enumerate(pick):
                img = load_shot_keyframe(movie_id, shot_idx)
                ax.imshow(img, extent=(x_cursor, x_cursor + 1, 0, 1))
                ax.add_patch(mpatches.Rectangle((x_cursor, 0), 1, 1, fill=False, edgecolor=cmap(sid % 20), linewidth=2))
                x_cursor += 1
            shown_scenes += 1
        ax.set_xlim(0, x_cursor)
        ax.set_ylim(0, 1)
        ax.set_title(f"{title} — {movie_id} (first {shown_scenes} scenes)")
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    print(f"GT scenes: {gt_scenes.max()+1} | Pred scenes: {pred_scenes.max()+1} | shots: {n}")
    print(f"Shot-level boundary F1 (this movie): {f1_score(y_true, pred, zero_division=0):.3f}")

## Boundary timeline

Plots per-shot boundary scores against ground-truth boundaries over a shot index window.

In [ ]:
def plot_boundary_timeline(movie_id: str, threshold: float = val_metrics["threshold"], window=(0, 200)):
    label_rows = parse_label_file(LABEL_DIR / f"{movie_id}.txt")
    y_true = np.array([r["is_boundary"] for r in label_rows], dtype=int)
    scores = predict_movie_scores(model, movie_id)
    pred = predict_boundaries(model, movie_id, threshold)

    n = min(len(y_true), len(scores), len(pred))
    y_true, scores, pred = y_true[:n], scores[:n], pred[:n]

    a, b = window
    b = min(b, n)
    x = np.arange(a, b)

    fig, axes = plt.subplots(2, 1, figsize=(14, 4), sharex=True)

    axes[0].plot(x, scores[a:b], color="C0", lw=1)
    axes[0].axhline(threshold, color="gray", ls="--", label=f"threshold={threshold:.3f}")
    axes[0].scatter(x[y_true[a:b] == 1], scores[a:b][y_true[a:b] == 1], c="green", s=20, label="GT boundary")
    axes[0].scatter(x[pred[a:b] == 1], scores[a:b][pred[a:b] == 1], c="red", marker="x", label="Pred boundary")
    axes[0].set_ylabel("Score")
    axes[0].legend(loc="upper right")
    axes[0].set_title(f"{movie_id} — boundary scores (shots {a}-{b})")

    gt_scene = labels_to_scene_ids(y_true)[a:b]
    pr_scene = labels_to_scene_ids(pred)[a:b]
    axes[1].imshow([gt_scene, pr_scene], aspect="auto", cmap="tab20", interpolation="nearest")
    axes[1].set_yticks([0, 1])
    axes[1].set_yticklabels(["GT scene id", "Pred scene id"])
    axes[1].set_xlabel("Shot index")
    plt.tight_layout()
    plt.show()

## Demo: multi-movie qualitative review

Runs comparison plots on a few test movies (edit `demo_movies` as needed).

In [ ]:
# Pick diverse test movies (edit as you like)
demo_movies = split["test"][:3]

for mid in demo_movies:
    if find_movie_keyframe_dir(mid) is None:
        print(f"Skip {mid} — no keyframes")
        continue
    plot_scene_comparison(mid, max_scenes=10, shots_per_scene=2)
    plot_boundary_timeline(mid, window=(0, 150))

## Scene video export

Builds an MP4 slideshow of keyframes with scene ID overlays — useful for presenting segmentation results in Colab.

In [ ]:
import imageio.v3 as iio
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path

def boundaries_to_scene_ids(boundary_flags: np.ndarray) -> np.ndarray:
    scene_ids = np.zeros(len(boundary_flags), dtype=int)
    sid = 0
    for i, b in enumerate(boundary_flags):
        scene_ids[i] = sid
        if b == 1:
            sid += 1
    return scene_ids

def load_keyframe(movie_id: str, shot_idx: int, which=(0, 1, 2)):
    """Load 1 or more keyframes for a shot."""
    kf_dir = find_movie_keyframe_dir(movie_id)
    frames = []
    for k in which:
        p = kf_dir / f"shot_{shot_idx:04d}_img_{k}.jpg"
        if p.exists():
            frames.append(np.array(Image.open(p).convert("RGB")))
    if not frames:
        frames.append(np.zeros((135, 240, 3), dtype=np.uint8))
    return frames

def annotate_frame(frame, text, bar_color=(255, 64, 64)):
    img = Image.fromarray(frame)
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 0, img.width, 28], fill=bar_color)
    draw.text((8, 6), text, fill=(255, 255, 255))
    return np.array(img)

def build_scene_video_frames(
    movie_id: str,
    pred_boundaries: np.ndarray,
    use_gt: bool = False,
    keyframes_per_shot=(1,),          # (1,) = middle only; (0,1,2) = all three
    max_shots: int | None = 300,      # limit length for Colab
    label_prefix: str = "Scene",
):
    label_rows = parse_label_file(LABEL_DIR / f"{movie_id}.txt")
    y = np.array([r["is_boundary"] for r in label_rows], dtype=int)
    flags = y if use_gt else pred_boundaries
    n = min(len(flags), len(pred_boundaries) if not use_gt else len(y))
    flags = flags[:n]

    scene_ids = boundaries_to_scene_ids(flags)
    if max_shots is not None:
        n = min(n, max_shots)
        scene_ids = scene_ids[:n]

    out_frames = []
    prev_scene = None

    for shot_idx in range(n):
        sid = int(scene_ids[shot_idx])
        is_new_scene = (prev_scene is not None) and (sid != prev_scene)
        prev_scene = sid

        kfs = (0, 1, 2) if keyframes_per_shot == (0, 1, 2) else (1,)
        for frame in load_keyframe(movie_id, shot_idx, which=kfs):
            text = f"{label_prefix} {sid} | shot {shot_idx:04d}"
            color = (255, 64, 64) if is_new_scene else (40, 40, 40)
            out_frames.append(annotate_frame(frame, text, bar_color=color))
            is_new_scene = False  # only highlight first frame of new scene

    return out_frames

Writes a list of RGB frames to MP4 using `imageio`.

In [ ]:
def save_keyframe_video(frames, out_path: Path, fps: float = 4.0):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    iio.imwrite(
        out_path,
        frames,
        fps=fps,
        codec="libx264",
        quality=8,
        macro_block_size=1,
    )
    print("Saved:", out_path, "| frames:", len(frames), "| duration ~", len(frames)/fps, "sec")
    return out_path

## Demo: export predicted-scene video

Renders and displays a short predicted-scene video for one test movie.

In [ ]:
movie_id = split["test"][0]   # pick any test movie with keyframes
threshold = val_metrics["threshold"]  # or BEST_THRESHOLD from boost run

scores = predict_movie_scores(model, movie_id)
pred = (scores >= threshold).astype(int)
if "scores_to_pred" in globals():
    pred = scores_to_pred(scores, threshold, min_len=4)

pred_frames = build_scene_video_frames(
    movie_id,
    pred_boundaries=pred,
    use_gt=False,
    keyframes_per_shot=(1,),     # middle keyframe per shot
    max_shots=250,               # trim for speed
    label_prefix="Pred scene",
)

out_mp4 = RESULTS_DIR / f"pred_scenes_{movie_id}.mp4"
save_keyframe_video(pred_frames, out_mp4, fps=4)

# Display in Colab
from IPython.display import Video, display
display(Video(str(out_mp4), embed=True, width=640))

## Ground-truth scene video (optional)

Same visualization pipeline using ground-truth boundaries instead of predictions.

In [ ]:
gt = np.array([r["is_boundary"] for r in parse_label_file(LABEL_DIR / f"{movie_id}.txt")], dtype=int)

gt_frames = build_scene_video_frames(movie_id, gt, use_gt=True, max_shots=250, label_prefix="GT scene")
pr_frames = build_scene_video_frames(movie_id, pred, use_gt=False, max_shots=250, label_prefix="Pred scene")

# Interleave short "GT clip" then "Pred clip" per scene range — or save two files:
save_keyframe_video(gt_frames, RESULTS_DIR / f"gt_scenes_{movie_id}.mp4", fps=4)
save_keyframe_video(pr_frames, RESULTS_DIR / f"pred_scenes_{movie_id}.mp4", fps=4)